In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import oracledb
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)
from sqlalchemy import create_engine
from sqlalchemy.dialects.oracle import NUMBER, TIMESTAMP, VARCHAR2

load_dotenv()
warnings.filterwarnings('ignore')

In [ ]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
CONNECT_STRING = '(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1522)(host=adb.sa-saopaulo-1.oraclecloud.com))(connect_data=(service_name=g253a13c6f2211a_dadosaiopslocaweb_low.adb.oraclecloud.com))(security=(ssl_server_dn_match=yes)))'

def conectar_oracle():
    return oracledb.connect(
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING,
        wallet_location=r"C:/opt/OracleCloud/Wallet_dadosAIOpsLocaweb",
        wallet_password=DB_PASSWORD
    )

In [ ]:
# Ler dados de uma view do banco de dados oracle
df = pd.read_sql_query("SELECT * FROM admin.TB_INCIDENTES_LOCAWEB", con=conectar_oracle())

In [ ]:
df.shape

In [ ]:
df.sample(50)

In [ ]:
print(df['DT_ABERTO'].min())
print("\n",df['DT_ABERTO'].max())


In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
df['TP_PRIORIDADE'].value_counts()

In [ ]:
df.dtypes

In [ ]:
df['DT_ABERTO'] = pd.to_datetime(df['DT_ABERTO'])
df['DT_ENCERRADO'] = pd.to_datetime(df['DT_ENCERRADO'])
df['DT_RESOLVIDO'] = pd.to_datetime(df['DT_RESOLVIDO'])

In [ ]:
incidentes_dia = df.groupby(df['DT_ABERTO'].dt.date)['TP_PRIORIDADE'].count()

In [ ]:
incidentes_dia

In [ ]:
print(f"A quantidade de dias no dataset é de: {df['DT_ABERTO'].dt.date.max() - df['DT_ABERTO'].dt.date.min()}, \
      \ndata inicial: {df['DT_ABERTO'].dt.date.min()} \
      \ndata final: {df['DT_ABERTO'].dt.date.max()}")

## Nem todas as datas estão preenchidas

In [ ]:
incidentes_dia.plot(legend=False)

## Muito ruído até 2025 

In [ ]:
data_limite = pd.to_datetime('2024-12-31')

In [ ]:
incidentes_dia = incidentes_dia.reset_index()

In [ ]:
incidentes_dia.columns

In [ ]:
incidentes_dia['DT_ABERTO'] = pd.to_datetime(incidentes_dia['DT_ABERTO']) 

In [ ]:
df_filtrado = incidentes_dia[incidentes_dia['DT_ABERTO'] > data_limite]
df_filtrado

In [ ]:
df_filtrado.plot(x ='DT_ABERTO',y='TP_PRIORIDADE')

# Será que os valores de 2023-2024 não tem valor algum? Verificar os que violaram KPI, se existir.

In [ ]:
df.head()

In [ ]:
df['FL_KPI_VIOLADO'].value_counts()

In [ ]:
cond1 = df['DT_ABERTO'] < data_limite
cond2 = df['FL_KPI_VIOLADO'] == 'SIM'

df_filtrado2 = df[cond1 & cond2]

## Só tem significância no PPT:

Você acaba de ganhar um slide pronto e de alto impacto para a sua apresentação executiva:

Insight Operacional: O Custo do Backlog Esquecido

A Descoberta: Durante a Análise Exploratória (EDA), identificamos incidentes que demoraram até 340 dias para serem encerrados, sendo alguns classificados apenas como "Falsos Positivos".

O Impacto: Esses "tickets órfãos" geram violações de OLA administrativas que mancham o indicador anual da equipe sem que haja um problema técnico real ocorrendo.

Ação Prática (Plano de Ação): Configurar na ferramenta de ITSM (ServiceNow, Jira, etc.) uma automação de "Timeout de Fila". Se um chamado P2/P3 ficar sem interação por mais de 48 horas, a ferramenta deve disparar um alerta para o Coordenador da equipe ou mudar o status para "Aguardando Usuário", congelando o relógio do OLA.

# Mais um Data Trimming

In [ ]:
df_filtrado.plot(x ='DT_ABERTO',y='TP_PRIORIDADE')

## Há uma diferença muito grande entre o volume de incidentes de Jan-Ago/2025 e Set-Dez/2025, justificando mais um Data Trimming para criar um modelo que possa predizer de forma precisa os incidentes atuais. No entanto, já podemos alertar de que caso ocorram mais mudanças muito bruscas como essas, pode ser necessário descartar os dados antigos e considerar novamente os mais recentes, identificando os marcos de tempo. 

In [ ]:
data_limite = '2025-08-31'

In [ ]:
df_filtrado = df_filtrado[df_filtrado['DT_ABERTO'] > data_limite]

In [ ]:
df_filtrado.shape

In [ ]:
df_filtrado.head()

# Feature Engineering

## Features de Calendário (Sazonalidade)
Estas variáveis ajudam a árvore de decisão a entender o contexto temporal (válidas tanto para D+1 quanto para D+7)

In [ ]:
df_filtrado['dia_semana'] = df_filtrado['DT_ABERTO'].dt.day_of_week

In [ ]:
def fim_semana(dia):
    if dia < 5:
        return 0
    else:
        return 1
    
df_filtrado['fl_fim_semana'] = df_filtrado['dia_semana'].apply(fim_semana)

In [ ]:
df_filtrado['dia_mes'] = df_filtrado['DT_ABERTO'].dt.day

In [ ]:
df_filtrado['mes'] = df_filtrado['DT_ABERTO'].dt.month

In [ ]:
df_filtrado.head(10)

## Features para D+1 (Inércia de Curto Prazo)
Para prever amanhã, o modelo pode e deve olhar para o que aconteceu hoje e ontem.

In [ ]:
colunas = { 
           'DT_ABERTO' : 'data',
           'TP_PRIORIDADE': 'vol_total'}

df_filtrado = df_filtrado.rename(columns= colunas)

In [ ]:
# Volume total de incidentes do dia anterior.
df_filtrado['vol_total_lag_1'] = df_filtrado['vol_total'].shift(1)

In [ ]:
# Volume de 2 e 3 dias atrás (captura a aceleração ou desaceleração do caos).
df_filtrado['vol_total_lag_2'] = df_filtrado['vol_total'].shift(2)
df_filtrado['vol_total_lag_3'] = df_filtrado['vol_total'].shift(3)

In [ ]:
# A média de volume dos últimos 3 e 7 dias. Suaviza os picos e mostra a tendência real para a árvore.
df_filtrado['media_movel_3d'] = df_filtrado['vol_total'].rolling(window=3, min_periods=1).mean()
df_filtrado['media_movel_7d'] = df_filtrado['vol_total'].rolling(window=7, min_periods=1).mean()

In [ ]:
df_filtrado.head()

# Features para D+7 (A Visão de Médio Prazo)
Atenção aqui: Se hoje é segunda-feira e você quer prever a próxima segunda-feira, você não pode usar dados de amanhã até domingo. O modelo só pode olhar do dia de hoje para trás.

In [ ]:
# O que aconteceu neste mesmo dia da semana, na semana passada.
df_filtrado['vol_total_lag_7'] = df_filtrado['vol_total'].shift(7)

In [ ]:
# Volume de exatas duas semanas atrás
df_filtrado['vol_total_lag_14'] = df_filtrado['vol_total'].shift(14)

In [ ]:
# A média móvel da semana, mas calculada com um atraso de 7 dias (para garantir que o modelo não espie o futuro).
df_filtrado['media_movel_7d_deslocada'] = df_filtrado['vol_total'].shift(7).rolling(window=7, min_periods=1).mean()

## Features de Pressão Operacional
Transformaremos linhas individuais em contagens diárias defasadas.

## Adicionando a contagem das prioridades do incidente para cada data 

In [ ]:
df.TP_PRIORIDADE.value_counts()

In [ ]:
df['DATA'] = df['DT_ABERTO'].dt.date

In [ ]:
prioridade_dia = df.groupby(['DATA', 'TP_PRIORIDADE']).size().reset_index(name='contagem')

In [ ]:
prioridade_dia

In [ ]:
prioridades = {
    '1 - Crítica': 1,
    '2 - Alta': 2,
    '3 - Média': 3,
    '4 - Baixa': 4,
    '5 - Muito Baixa': 5
}

def converter_prioridades(df, coluna='TP_PRIORIDADE'):
    df[coluna] = df[coluna].map(prioridades)
    return df

converter_prioridades(prioridade_dia)

In [ ]:
tabela = prioridade_dia.pivot(index='DATA', columns='TP_PRIORIDADE', values='contagem').fillna(0)

In [ ]:
tabela.head()

In [ ]:
tabela.reset_index(inplace=True)

In [ ]:
tabela.dtypes

In [ ]:
tabela['DATA'] = pd.to_datetime(tabela['DATA'])

In [ ]:
tabela = tabela[tabela['DATA'] > data_limite]

In [ ]:
tabela.shape

In [ ]:
df_filtrado.shape

In [ ]:
tabela.rename(columns={'DATA':'data'}, inplace=True)

In [ ]:
tabela.head()

In [ ]:
df_filtrado.head()

In [ ]:
df_filtrado.set_index('data',inplace=True)
tabela.set_index('DATA', inplace=True)

In [ ]:
df_final = df_filtrado.merge(tabela, on='data', how='left')

In [ ]:
df_final.dtypes

In [ ]:
prioridades = {
    1 : 'vol_p1',
    2 : 'vol_p2',
    3 : 'vol_p3',
    4 : 'vol_p4',
    5 : 'vol_p5'
}

df_final.rename(columns=prioridades, inplace=True)

### Pressão por Prioridade
Se a equipe de TI foi inundada por chamados de baixa prioridade hoje, o cansaço e o backlog podem estourar o volume crítico amanhã. 

In [ ]:
colunas_prioridade = ['vol_p2', 'vol_p3', 'vol_p4', 'vol_p5']
lags = [1, 7]

for col in colunas_prioridade:
    for lag in lags:
        df_final[f'{col}_lag_{lag}'] = df_final[col].shift(lag)

print("Colunas criadas:")
print([col for col in df_final.columns if 'lag' in col])

### Carga de Monitoramento
O dicionário aponta que incidentes de monitoramento geram volume sem intervenção humana. Um pico aqui hoje pode indicar uma instabilidade sistêmica silenciosa que vai gerar chamados manuais amanhã.

In [ ]:
tabela = df.groupby(['DATA','TP_ABERTO_POR']).size().reset_index(name='contagem')

In [ ]:
tabela.rename(columns={'DATA':'data'}, inplace=True)

In [ ]:
tabela = tabela.pivot(index='data', columns='TP_ABERTO_POR', values='contagem').fillna(0)

In [ ]:
tabela.rename(columns={'Manual':'vol_aberto_manual', 'Monitoramento': 'vol_aberto_monitoramento'}, inplace=True)

In [ ]:
tabela.reset_index(inplace=True)

In [ ]:
tabela['data'] = pd.to_datetime(tabela['data'])

In [ ]:
tabela.dtypes

In [ ]:
df_final = df_final.merge(tabela, on='data', how='left')

In [ ]:
df_final.dtypes

## Focos de Incêndio (Top 5)
Escolha as 5 categoria (nm_grupo_designado) e as 5 categorias (nm_categoria) que mais recebem chamados no geral. Crie features diárias do tipo vol_team14_lag ou vol_cat_banco_de_dados_lag. Isso ensina ao modelo onde o problema costuma nascer antes de se espalhar.

### Top 7 equipes mais requisitadas

In [ ]:
equipes = df[df['DT_ABERTO'] > data_limite].groupby('NM_GRUPO_DESIGNADO')['DATA'].count().sort_values(ascending=False)
equipes.head(17)

### Top 6 Categorias mais frequentes

In [ ]:
categoria = df[df['DT_ABERTO'] > data_limite].groupby('NM_CATEGORIA')['DATA'].count().sort_values(ascending=False)
categoria.head(114)

In [ ]:
df_final.head()

In [ ]:
df_final.set_index('data', inplace=True)

In [ ]:
# 1. Definindo suas listas de interesse (Corrigi os pequenos erros de digitação)
categorias_alvo = ['Nao Informado', 'cat71', 'cat77', 'cat76', 'cat73', 'cat85']
times_alvo = ['Team14', 'Team05', 'Team11', 'Team12', 'Team09', 'Team10', 'Team03']

# 2. Garantindo que o 'df' bruto tenha uma coluna apenas com a data (sem as horas)
df['DATA'] = pd.to_datetime(df['DT_ABERTO']).dt.floor('D')

# 3. Criando as matrizes de contagem diária
# Conta quantas vezes cada categoria e cada time apareceu por dia
cat_counts = pd.crosstab(df['DATA'], df['NM_CATEGORIA'])
team_counts = pd.crosstab(df['DATA'], df['NM_GRUPO_DESIGNADO'])

# Filtra apenas as colunas que estão nas suas listas (evita erros se uma categoria não existir)
colunas_cat_reais = [c for c in categorias_alvo if c in cat_counts.columns]
colunas_team_reais = [t for t in times_alvo if t in team_counts.columns]

cat_counts = cat_counts[colunas_cat_reais]
team_counts = team_counts[colunas_team_reais]

# 4. Juntando as contagens no seu df_final
# IMPORTANTE: O df_final precisa estar com a data no index para o .join() funcionar corretamente
df_final = df_final.join(cat_counts, how='left').fillna(0)
df_final = df_final.join(team_counts, how='left').fillna(0)

# 5. Criando os Lags (A Máquina do Tempo para o XGBoost)
todas_novas_colunas = colunas_cat_reais + colunas_team_reais
lags = [1, 7]

for col in todas_novas_colunas:
    # Renomeando e criando o lag para manter o padrão 'vol_nome_lag_x'
    nome_limpo = col.replace(' ', '_').lower() # Ex: 'Não Informado' vira 'não_informado'
    
    for lag in lags:
        df_final[f'vol_{nome_limpo}_lag_{lag}'] = df_final[col].shift(lag)
        
    # Opcional, mas recomendado: Excluir a contagem do dia "atual" para evitar Data Leakage
    # O modelo D+1/D+7 só pode ver os dados de "ontem" e da "semana passada"
    df_final.drop(columns=[col], inplace=True)

# Remove os dias iniciais que ficaram com NaN devido ao lag_7
df_final.dropna(inplace=True)

print("Colunas de contexto criadas com sucesso:")
print([c for c in df_final.columns if 'lag' in c])

In [ ]:
df_final.dtypes.reset_index()

In [ ]:
df_final.head()

In [ ]:
# 1. Criando os lags que faltaram (Abertura)
colunas_abertura = ['vol_aberto_manual', 'vol_aberto_monitoramento']
lags = [1, 7]

for col in colunas_abertura:
    for lag in lags:
        df_final[f'{col}_lag_{lag}'] = df_final[col].shift(lag)

# 2. A "Lista Negra" do Data Leakage (Variáveis do dia atual que não podemos mostrar ao modelo)
colunas_vazamento = [
    'vol_p1', 'vol_p2', 'vol_p3', 'vol_p4', 'vol_p5',
    'vol_aberto_manual', 'vol_aberto_monitoramento'
]

# 3. Dropando as colunas proibidas
df_final.drop(columns=colunas_vazamento, inplace=True, errors='ignore')

# 4. Limpando os novos valores nulos (NaN) gerados pelo shift de 7 dias
df_final.dropna(inplace=True)

print("Drop realizado com sucesso. Variáveis retidas para o modelo:")
print(df_final.columns.tolist())

# Treinamento do Modelo XGBoost

In [ ]:
import xgboost as xgb

In [ ]:
coluna_alvo = 'vol_total'
X = df_final.drop(columns=[coluna_alvo])
y = df_final[coluna_alvo]

In [ ]:
tamanho_teste = 30
X_train, X_test = X.iloc[:-tamanho_teste], X.iloc[-tamanho_teste:]
y_train, y_test = y.iloc[:-tamanho_teste], y.iloc[-tamanho_teste:]

print(f"Treinando com {len(X_train)} dias. Testando com os últimos {len(X_test)} dias.")

In [ ]:
modelo_xgb = xgb.XGBRegressor(
    n_estimators=100,
    max_depth = 4,
    learning_rate = 0.05,
    random_state = 23
)

In [ ]:
modelo_xgb.fit(X_train, y_train)

In [ ]:
previsoes = modelo_xgb.predict(X_test)

In [ ]:
mae_xgb = mean_absolute_error(y_test, previsoes)
rmse_xgb = np.sqrt(mean_squared_error(y_test, previsoes))

In [ ]:
print("\n--- Performance do XGBoost (Modelo A: D+1) ---")
print(f"MAE: {mae_xgb:.2f} incidentes")
print(f"RMSE: {rmse_xgb:.2f} incidentes")

plt.figure(figsize=(12,5))
plt.plot(y_test.index, y_test.values, label='Volume Real', color='blue', marker='o')
plt.plot(y_test.index, previsoes, label='Previsão XGBoost', color='red', linestyle='--', marker='x')
plt.title("XGBoost - Validação D+1 (Últimos 30 dias)")
plt.ylabel("Volume de Incidentes")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
importancias = modelo_xgb.feature_importances_
nomes_das_features = X_train.columns

In [ ]:
df_importancia = pd.DataFrame({
    'Feature': nomes_das_features,
    'Importancia': importancias
})

In [ ]:
df_importancia = df_importancia.sort_values(by='Importancia', ascending=True).tail(15)

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(df_importancia['Feature'], df_importancia['Importancia'], color ='#1f77b4', edgecolor='black')
plt.title('Top 15 Fatores que mais Influenciam o Volume de Incidentes (D+1)')
plt.xlabel('Peso de Importância (F-Score)')
plt.ylabel('Variáveis de Contexto')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## Investigando Team 05

In [ ]:
# Analisando o que o Team 05 faz
df_team05 = df[df['NM_GRUPO_DESIGNADO'] == 'Team05']

print("--- Top 5 Categorias do Team 05 ---")
print(df_team05['NM_CATEGORIA'].value_counts().head(5))

print("\n--- Distribuição de Prioridades do Team 05 ---")
print(df_team05['TP_PRIORIDADE'].value_counts())

In [ ]:
## Insights

# Treinando D+7 

In [ ]:
# 1. Identificando as colunas proibidas para D+7 (Tudo que for menor que 7 dias)
colunas_para_dropar = [col for col in df_final.columns if 'lag_1' in col or 'lag_2' in col or 'lag_3' in col]
colunas_para_dropar.append('media_movel_3d') # Também vaza dados da mesma semana

In [ ]:
# 2. Criando a base exclusiva para o D+7
df_d7 = df_final.drop(columns=colunas_para_dropar, errors='ignore')

In [ ]:
# 3. Separando X e y
coluna_alvo = 'vol_total'
X_d7 = df_d7.drop(columns=[coluna_alvo])
y_d7 = df_d7[coluna_alvo]

In [ ]:
# 4. Split Temporal (Últimos 30 dias para teste)
tamanho_teste = 30
X_train_d7, X_test_d7 = X_d7.iloc[:-tamanho_teste], X_d7.iloc[-tamanho_teste:]
y_train_d7, y_test_d7 = y_d7.iloc[:-tamanho_teste], y_d7.iloc[-tamanho_teste:]

print(f"Colunas usadas no treino D+7: {len(X_train_d7.columns)}")

In [ ]:
# 5. Treinando o XGBoost D+7
modelo_xgb_d7 = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=23
)

In [ ]:
modelo_xgb_d7.fit(X_train_d7, y_train_d7)

In [ ]:
# 6. Previsões e Métricas
previsoes_d7 = modelo_xgb_d7.predict(X_test_d7)

mae_d7 = mean_absolute_error(y_test_d7, previsoes_d7)
rmse_d7 = np.sqrt(mean_squared_error(y_test_d7, previsoes_d7))

print("\n--- Performance do XGBoost (Modelo A: D+7) ---")
print(f"MAE: {mae_d7:.2f} incidentes")
print(f"RMSE: {rmse_d7:.2f} incidentes")

In [ ]:
# 7. Plot Rápido
plt.figure(figsize=(12, 4))
plt.plot(y_test_d7.index, y_test_d7.values, label='Volume Real', color='blue', marker='o')
plt.plot(y_test_d7.index, previsoes_d7, label='Previsão D+7', color='green', linestyle='--', marker='x')
plt.title("XGBoost - Validação D+7 (Últimos 30 Dias)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# Feriados: Podemos ver que há uma zona nebulosa próxima ao Natal, vamos acrescentar e ver se há uma melhora!

In [ ]:
import holidays

# 1. Mapeando os feriados de 2025 (incluindo os feriados estaduais do AM para máxima precisão local)
feriados_2025 = holidays.country_holidays('BR', subdiv='AM', years=2025)

# Se a data estiver no index, usamos df.index. Se estiver numa coluna 'DATA', troque df.index por df['DATA']
datas = df_d7.index 

# 2. Criando a Flag de Feriado (1 = Sim, 0 = Não)
df_d7['fl_feriado'] = datas.map(lambda x: 1 if x in feriados_2025 else 0)

# 3. Criando a Flag de Véspera de Feriado (1 = Sim, 0 = Não)
df_d7['fl_vespera_feriado'] = datas.map(
    lambda x: 1 if (x + pd.Timedelta(days=1)) in feriados_2025 else 0
)

print("Novas colunas adicionadas:")
print(df_d7[['fl_feriado', 'fl_vespera_feriado']].value_counts())

In [ ]:
# 3. Separando X e y
coluna_alvo = 'vol_total'
X_d7 = df_d7.drop(columns=[coluna_alvo])
y_d7 = df_d7[coluna_alvo]

In [ ]:
# 4. Split Temporal (Últimos 30 dias para teste)
tamanho_teste = 30
X_train_d7, X_test_d7 = X_d7.iloc[:-tamanho_teste], X_d7.iloc[-tamanho_teste:]
y_train_d7, y_test_d7 = y_d7.iloc[:-tamanho_teste], y_d7.iloc[-tamanho_teste:]

print(f"Colunas usadas no treino D+7: {len(X_train_d7.columns)}")

In [ ]:
# 5. Treinando o XGBoost D+7
modelo_xgb_d7 = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    random_state=23
)

In [ ]:
modelo_xgb_d7.fit(X_train_d7, y_train_d7)

In [ ]:
# 6. Previsões e Métricas
previsoes_d7 = modelo_xgb_d7.predict(X_test_d7)

mae_d7 = mean_absolute_error(y_test_d7, previsoes_d7)
rmse_d7 = np.sqrt(mean_squared_error(y_test_d7, previsoes_d7))

print("\n--- Performance do XGBoost (Modelo A: D+7) ---")
print(f"MAE: {mae_d7:.2f} incidentes")
print(f"RMSE: {rmse_d7:.2f} incidentes")

In [ ]:
# 7. Plot Rápido
plt.figure(figsize=(12, 4))
plt.plot(y_test_d7.index, y_test_d7.values, label='Volume Real', color='blue', marker='o')
plt.plot(y_test_d7.index, previsoes_d7, label='Previsão D+7', color='green', linestyle='--', marker='x')
plt.title("XGBoost - Validação D+7 (Últimos 30 Dias)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
importancias = modelo_xgb_d7.feature_importances_
nomes_das_features = X_train_d7.columns

In [ ]:
df_importancia = pd.DataFrame({
    'Feature': nomes_das_features,
    'Importancia': importancias
})

In [ ]:
df_importancia = df_importancia.sort_values(by='Importancia', ascending=True).tail(15)

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(df_importancia['Feature'], df_importancia['Importancia'], color ='#1f77b4', edgecolor='black')
plt.title('Top 15 Fatores que mais Influenciam o Volume de Incidentes (D+7)')
plt.xlabel('Peso de Importância (F-Score)')
plt.ylabel('Variáveis de Contexto')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## Investigando Team 10

In [ ]:
# Analisando o que o Team 05 faz
df_team10 = df[df['NM_GRUPO_DESIGNADO'] == 'Team10']

print("--- Top 5 Categorias do Team 10 ---")
print(df_team10['NM_CATEGORIA'].value_counts().head(5))

print("\n--- Distribuição de Prioridades do Team 10 ---")
print(df_team10['TP_PRIORIDADE'].value_counts())

## Insights

# Modelo de Classificação do OLA

In [ ]:
# Filtrando o cenário atual e as prioridades alvo
df_risco = df[
    (df['DT_ABERTO'] >= '2025-09-01') &
    (df['TP_PRIORIDADE'].isin(['2 - Alta', '3 - Média']))
].copy()

In [ ]:
# Criando features de momento
df_risco['data'] = pd.to_datetime(df_risco['DT_ABERTO'])
df_risco['hora_abertura'] = df_risco['data'].dt.hour
df_risco['dia_semana'] = df_risco['data'].dt.day_of_week

In [ ]:
# Transformando a coluna alvo (Target)
# 'SIM' vira 1 (Violou), 'NÃO' vira 0 (Dentro do prazo)

df_risco['alvo_kpi'] = df_risco['FL_KPI_VIOLADO'].map({'SIM': 1, 'NAO': 0})

In [ ]:
# Dropando o Futuro e identificadores inúteis
colunas_proibidas = [
    'CD_INCIDENTE', 'DT_ABERTO', 'DT_RESOLVIDO', 'DT_ENCERRADO', 'DATA',
    'NR_DURACAO', 'CD_FECHAMENTO', 'TP_STATUS', 'DS_RESUMIDA', 
    'FL_KPI_VIOLADO', 'FL_KPI', 'NM_ITEM_CONFIGURACAO', 'CD_INCIDENTE_PAI'
]

df_risco.drop(columns=colunas_proibidas, inplace=True, errors='ignore')

In [ ]:
# Checando o desbalanceamento
print(f"Total de Chamados (P2/P3): {len(df_risco)}")
print("\nDistribuição do Risco (0 = OK, 1 = Violado):")
print(df_risco['alvo_kpi'].value_counts(normalize=True)*100)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

df_baseline = df_risco.copy()

In [ ]:
# 4. Transformação Categórica (One-Hot Encoding)
df_baseline = pd.get_dummies(df_baseline, columns=['NM_CATEGORIA', 'NM_GRUPO_DESIGNADO'], drop_first=True)

In [ ]:
# 5. Separação de Variáveis
X_base = df_baseline.drop(columns=['alvo_kpi'])
y_base = df_baseline['alvo_kpi']

In [ ]:
X_base = X_base.select_dtypes(include=['int32', 'int64', 'float32', 'float64', 'bool', 'uint8'])

In [ ]:
# Calculando a penalidade matemática para focar nos 4%
peso = y_base.value_counts()[0] / y_base.value_counts()[1]

In [ ]:
# 6. Split Temporal (Os últimos 20% dos chamados simulam o futuro da operação)
X_train, X_test, y_train, y_test = train_test_split(X_base, y_base, test_size=0.2, shuffle=False)

print(f"Treinando com {len(X_train)} chamados. Testando com {len(X_test)} chamados.")
print(f"Peso aplicado na classe minoritária: {peso:.2f}\n")

In [ ]:
# 7. Treinamento do XGBoost Classifier (Baseline)
modelo_clf_base = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=peso,
    random_state=42
)

In [ ]:
modelo_clf_base.fit(X_train, y_train)

In [ ]:
# 8. Avaliação
previsoes_base = modelo_clf_base.predict(X_test)

print("--- Relatório de Classificação (Baseline) ---")
print(classification_report(y_test, previsoes_base, target_names=['No Prazo (0)', 'Violado (1)']))

In [ ]:
# 9. Matriz de Confusão Visual
cm = confusion_matrix(y_test, previsoes_base)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Prazo', 'Violado'])
disp.plot(cmap='Blues', values_format='d')
plt.title("Matriz de Confusão - Guardião do OLA (Baseline)")
plt.show()

Os números dessa matriz de confusão contam uma história fascinante sobre como a Inteligência Artificial lida com riscos quando é penalizada.

Respondendo diretamente às suas perguntas:

O modelo detectou corretamente 363 chamados que iam estourar o OLA (esses são os Verdadeiros Positivos no quadrante inferior direito).

O Recall foi de 0.91 (91%).

Do ponto de vista de negócio, esse número é assustadoramente bom para um baseline. Significa que, no "minuto zero", o modelo conseguiu interceptar 91% de todas as bombas que iam explodir no colo do Team 05 e das outras equipes. Um analista humano de Nível 1 jamais conseguiria prever isso apenas olhando para a categoria e a hora de abertura.

O Custo da Segurança: A Síndrome do "Falso Alarme"
Como aplicamos aquele peso altíssimo (scale_pos_weight), nós basicamente dissemos ao algoritmo: "Errar um chamado que vai estourar é 24 vezes pior do que dar um falso alarme".

O XGBoost obedeceu fielmente e se tornou paranoico.
Olhe para o número 1735 no quadrante superior direito (Falsos Positivos). O modelo apontou o dedo para 1.735 chamados dizendo "Isso vai estourar!", mas no fim das contas a equipe conseguiu resolver no prazo. É por isso que a sua Precision (Precisão) foi de apenas 17%.

O Risco para o Produto (Visão PO): Se você colocar esse modelo em produção hoje, a equipe de TI sofrerá de "Alert Fatigue" (Fadiga de Alarme). Eles vão receber tantos alertas falsos que começarão a ignorar o modelo.

O Próximo Passo: Dando "Visão" ao Modelo
Por que o baseline gerou tantos alarmes falsos? Porque ele está cego para o caos.
Ele sabe que um chamado da cat76 entrou às 14h de uma terça-feira, mas ele não sabe se a equipe passou o dia de ontem apagando incêndios ou se a fila está limpa.

É aqui que a sua ideia de injetar a pressão operacional (os lags) vai brilhar. Ao cruzar os dados do minuto zero com o histórico do dia anterior, nossa hipótese é que o modelo aprenderá a diferenciar uma "terça-feira tranquila" de uma "terça-feira pós-feriado caótica", reduzindo os alarmes falsos sem perder a taxa de 91% de acerto.

In [ ]:
# 1. Isolando a "fotografia da pressão operacional" do seu df_final (Apenas Lags)
colunas_pressao = [col for col in df_final.columns if 'lag' in col or 'media_movel' in col]
df_pressao_historica = df_final[colunas_pressao].copy()

# 2. Garantindo que a data do df_risco seja apenas o "dia" para fazer o join correto
df_risco['DATA_DIA'] = df_risco['data'].dt.floor('D')

# 3. Injetando a pressão operacional no minuto zero do chamado
df_risco = df_risco.merge(df_pressao_historica, left_on='DATA_DIA', right_index=True, how='left')
df_risco.drop(columns=['DATA_DIA'], inplace=True) # Limpando a coluna auxiliar

# 4. Dropando chamados que caíram nos primeiros dias de setembro (Warm-up period com NaN nos lags)
df_risco.dropna(inplace=True)

# 5. Tratamento de Variáveis Categóricas Restantes (One-Hot Encoding)
# Como o modelo precisa de números, vamos transformar Categoria e Grupo em colunas 0 e 1
colunas_categoricas = ['NM_CATEGORIA', 'NM_GRUPO_DESIGNADO']
df_risco = pd.get_dummies(df_risco, columns=colunas_categoricas, drop_first=True)

In [ ]:
# 1. Separando X e y do dataset completo (que já contém os lags de pressão)
X_full = df_risco.drop(columns=['alvo_kpi'])
y_full = df_risco['alvo_kpi']

# 2. A "Vassoura Definitiva" para garantir que apenas matemática passe
X_full = X_full.select_dtypes(include=['int32', 'int64', 'float32', 'float64', 'bool', 'uint8'])

# 3. Split Temporal (Mantendo os últimos 20% para teste)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_full, y_full, test_size=0.2, shuffle=False)

# 4. Calculando a penalidade baseada no treino
peso_full = y_train_f.value_counts()[0] / y_train_f.value_counts()[1]

print(f"Treinando modelo COMPLETO com {len(X_train_f.columns)} features.")

# 5. Treinamento
modelo_clf_full = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=peso_full,
    random_state=42
)

modelo_clf_full.fit(X_train_f, y_train_f)

# 6. Previsão e Avaliação
previsoes_full = modelo_clf_full.predict(X_test_f)

print("\n--- Relatório de Classificação (Guardião do OLA com Pressão Operacional) ---")
print(classification_report(y_test_f, previsoes_full, target_names=['No Prazo (0)', 'Violado (1)']))

# 7. Matriz Visual
cm_full = confusion_matrix(y_test_f, previsoes_full)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_full, display_labels=['No Prazo', 'Violado'])
disp.plot(cmap='Greens', values_format='d')
plt.title("Matriz de Confusão - Guardião (Com Pressão)")
plt.show()

## O Trade-off de Ouro para Negócios

Os Falsos Positivos despencaram de 1.735 para 989. Conseguimos blindar a equipe de TI contra quase 750 alarmes falsos, garantindo que o modelo mantenha credibilidade na operação diária.

A acurácia global saltou para 83% e o F1-Score da classe minoritária (a mais difícil) subiu de 0.29 para 0.32.

O Recall ajustou de 0.91 para 0.79. O classificador agora deixa passar cerca de 2 em cada 10 violações, mas intercepta cirurgicamente as outras 8 com o dobro de assertividade. Para uma anomalia que representa apenas 4% da base, prever 79% dos estouros de prazo no exato minuto da abertura do chamado é um feito de alto nível técnico.

# Modelo de Clusterização - Entendendo o que já aconteceu.

## O Plano de Ação (Data Prep & Elbow Method)
Para essa análise post-mortem (investigação de causa raiz), nós podemos usar dados do futuro (como a duração do chamado ou se ele violou o KPI), pois não estamos prevendo nada, estamos apenas tentando entender o que já aconteceu.

In [ ]:
df.columns

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Filtrando o cenário atual (Pós-Setembro)
df_cluster = df[df['DT_ABERTO'] >= '2025-09-01'].copy()

# 2. Selecionando as variáveis investigativas 
features_analise = ['TP_PRIORIDADE', 'NM_GRUPO_DESIGNADO', 'NM_CATEGORIA', 'FL_KPI_VIOLADO', 'NR_DURACAO', 'TP_ABERTO_POR']
df_features = df_cluster[features_analise].dropna()

# 3. Transformando texto em números (One-Hot Encoding)
X_cluster = pd.get_dummies(df_features, drop_first=True)

# 4. A REGRA DE OURO: Normalização (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Base pronta para Clusterização: {X_scaled.shape[0]} incidentes e {X_scaled.shape[1]} dimensões.")

In [ ]:
# 5. Método do Cotovelo (Testando de 2 a 8 clusters)
inercia = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inercia.append(kmeans.inertia_)

# 6. Plotando o Cotovelo
plt.figure(figsize=(9, 5))
plt.plot(K_range, inercia, marker='o', linestyle='--', color='#d62728', linewidth=2)
plt.title('Método do Cotovelo (Elbow Method) - Descobrindo o Número Ideal de Clusters')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (Distância Interna dos Grupos)')
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

# A Leitura do Cotovelo e a Decisão de Produto
Observe o salto entre o K=3 e o K=4. A linha cai de forma bem acentuada (de 1.23 para 1.21). No entanto, ao passar do K=4 para o K=5, a inclinação perde a força e começa a achatar levemente. Matematicamente, o K=4 é o ponto onde o ganho de informação adicional começa a diminuir.

In [ ]:
# 1. Aplicando o K-Means definitivo com K=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df_features['Cluster'] = kmeans_final.fit_predict(X_scaled)

In [ ]:
# 2. Descobrindo o "DNA" de cada Cluster
# Vamos agrupar os dados e pegar a MÉDIA do tempo e o valor MAIS COMUM (moda) dos textos
perfil_clusters = df_features.groupby('Cluster').agg({
    'NR_DURACAO': 'mean', # Média de tempo em segundos
    'TP_PRIORIDADE': lambda x: x.mode()[0] if not x.mode().empty else 'N/A',
    'NM_GRUPO_DESIGNADO': lambda x: x.mode()[0] if not x.mode().empty else 'N/A',
    'NM_CATEGORIA': lambda x: x.mode()[0] if not x.mode().empty else 'N/A',
    'FL_KPI_VIOLADO': lambda x: x.mode()[0] if not x.mode().empty else 'N/A',
    'TP_ABERTO_POR': lambda x: x.mode()[0] if not x.mode().empty else 'N/A'
}).reset_index()

In [ ]:
# 3. Contando o tamanho de cada grupo (para ver quem domina a operação)
tamanho_clusters = df_features['Cluster'].value_counts().reset_index()
tamanho_clusters.columns = ['Cluster', 'Qtd_Incidentes']

In [ ]:
# 4. Unindo e formatando a tabela final
resultado_final = pd.merge(tamanho_clusters, perfil_clusters, on='Cluster')

In [ ]:
# Convertendo a duração média de segundos para horas para facilitar a leitura no PPT
resultado_final['NR_DURACAO_HORAS'] = (resultado_final['NR_DURACAO'] / 3600).round(2)
resultado_final.drop(columns=['NR_DURACAO'], inplace=True)

print("--- O DNA da Operação (4 Perfis de Incidentes) ---")
print(resultado_final.to_string(index=False))

## 1. Cluster 3: "O Ruído Automatizado" (A Ilusão de Volume)
* O Cenário: É o dono absoluto da base (86.903 chamados). São alertas de prioridade Baixa (P4), abertos por Monitoramento, sem categoria definida ("Não Informado") e que o Team 14 resolve em menos de 3 horas.

* O Diagnóstico de Negócio: Isso não é gestão de incidentes, é fadiga de alerta (Alert Fatigue). As ferramentas de monitoramento (como Zabbix ou Prometheus) estão mal calibradas, disparando alertas inúteis que inflam artificialmente o volume da operação e poluem as métricas da diretoria.

* Plano de Ação: Criar um Épico para a engenharia de confiabilidade (SRE) refinar os gatilhos (limiares) de monitoramento do Team 14.

## 2. Cluster 2: "O Gargalo Crítico" (O Coração do Problema)
* O Cenário: Aqui está ele novamente, o nosso velho conhecido Team 05. Incidentes de prioridade Alta (P2), focados na cat76, demorando em média 13 horas!

* O Diagnóstico de Negócio: O K-Means acabou de validar perfeitamente o que o XGBoost descobriu. A inteligência artificial encontrou a raiz do caos sem que disséssemos nada a ela. A cat76 gerida pelo Team 05 é o maior risco sistêmico da Locaweb.

* Plano de Ação: Congelar novas features dessa categoria e focar exclusivamente em estabilização e refatoração técnica.

## 3. Cluster 0: "O Buraco Negro Manual"
* O Cenário: Apenas 3.500 chamados, mas abertos manualmente, caindo para o Team 11 na cat85. A duração média é assustadora: 175 horas (mais de 7 dias pendentes).

* O Diagnóstico de Negócio: Como são chamados manuais, isso aqui é dor direta do usuário final. O Team 11 tem um processo completamente quebrado de atendimento ou está lidando com uma categoria (cat85) que exige aprovações de terceiros/fornecedores, o que trava o fluxo por uma semana inteira.

* Plano de Ação: Revisão imediata do fluxo de valor (Value Stream Mapping) do Team 11 para entender onde o chamado fica parado nesses 7 dias.

## 4. Cluster 1: "O Ralo de Produtividade"
* O Cenário: Team 12 recebendo chamados de prioridade Média (P3), vindos do monitoramento, mas totalmente como "Não Informado".

* O Diagnóstico de Negócio: Idêntico ao problema que tínhamos mapeado no Team 10 anteriormente. Monitoramento criando chamado cego de prioridade média gera desperdício de tempo de triagem.

In [ ]:
# 1. Resgatando os dados reais do Modelo D+7
# y_test_d7 possui os volumes reais e previsoes_d7 possui o que o modelo previu
df_volume = pd.DataFrame({
    'DATA': y_test_d7.index,          # Pega as datas dos últimos 30 dias
    'VOLUME_REAL': y_test_d7.values,  # O volume que de fato aconteceu
    'PREVISAO_D7': previsoes_d7       # O que a IA previu
})

# 2. Adicionando a previsão do D+1
df_volume['PREVISAO_D1'] = previsoes 

# 3. Arredondando porque não existe meio chamado
df_volume['PREVISAO_D7'] = df_volume['PREVISAO_D7'].round().astype(int)
df_volume['PREVISAO_D1'] = df_volume['PREVISAO_D1'].round().astype(int)

# Opcional: Se a 'DATA' não virou coluna e ficou no índice, podemos forçar o reset:
if 'DATA' not in df_volume.columns:
    df_volume.reset_index(inplace=True)
    df_volume.rename(columns={'index': 'DATA'}, inplace=True)

print("Tabela consolidada D+1 e D+7 criada com sucesso!")
print(df_volume.head())

In [ ]:
df_volume

In [ ]:
# 1. Resgatando os dados legíveis usando os índices do teste
indices_teste = X_test_f.index
colunas_dashboard = ['CD_INCIDENTE', 'DT_ABERTO', 'TP_PRIORIDADE', 'NM_CATEGORIA', 'NM_GRUPO_DESIGNADO']

# Puxando do 'df' original
df_alertas = df.loc[indices_teste, colunas_dashboard].copy()

# 2. Injetando o cérebro do modelo de Machine Learning
df_alertas['RISCO_SLA_PREVISTO'] = previsoes_full # O que o modelo disse (1 ou 0)
df_alertas['STATUS_REAL'] = y_test_f              # O que realmente aconteceu (apenas para conferência)

# 3. Filtrando a "Fila de Fogo": Apenas chamados que o modelo previu que VÃO ESTOURAR (Risco = 1)
df_alertas_criticos = df_alertas[df_alertas['RISCO_SLA_PREVISTO'] == 1].copy()

# 4. Atendendo a Fase 3 (GenAI Mock) - O Diferencial para a Banca
# Criamos uma função que simula a resposta do Groq/Llama3 baseada no contexto do chamado
def mock_genai_prescritivo(linha):
    equipe = linha['NM_GRUPO_DESIGNADO']
    categoria = linha['NM_CATEGORIA']
    
    if equipe == 'Team05' and categoria == 'cat76':
        return f"ALERTA GENAI: 85% de risco de quebra no {equipe} devido a gargalo histórico na {categoria}. Recomendação: 1. Escalone para Nível 2 imediatamente. 2. Congele mudanças não emergenciais no produto afetado."
    elif equipe == 'Team11':
        return f"ALERTA GENAI: Incidente no {equipe} corre alto risco de estagnação. Recomendação: 1. Acione o fornecedor/terceiro responsável. 2. Atualize o usuário final para evitar reincidência de abertura."
    elif equipe == 'Team14':
        return f"INFO GENAI: Falso positivo provável no {equipe} gerado por monitoramento. Recomendação: Validar log do Zabbix/Prometheus antes de alocar analista humano."
    else:
        return f"ALERTA GENAI: Risco de quebra de OLA no {equipe} ({categoria}). Recomendação: Priorizar na fila do Kanban e notificar o Coordenador de Turno."

# Aplicando a IA Generativa (Simulada)
df_alertas_criticos['TEXTO_PRESCRITIVO_LLM'] = df_alertas_criticos.apply(mock_genai_prescritivo, axis=1)

# Formatando a data para o banco
df_alertas_criticos['DT_ABERTO'] = pd.to_datetime(df_alertas_criticos['DT_ABERTO'])

# 5. Exportando para CSV e validando
df_alertas_criticos.to_csv('df_alertas_kpi.csv', index=False)

print(f"Foram gerados {len(df_alertas_criticos)} Alertas Críticos de OLA para o Dashboard.")
print(df_alertas_criticos[['NM_GRUPO_DESIGNADO', 'NM_CATEGORIA', 'TEXTO_PRESCRITIVO_LLM']].head())

In [ ]:
# A sua tabela gerada no passo de Clusterização é a base perfeita para a Matriz de Risco
df_matriz_risco = resultado_final.copy()

# Renomeando para ficar bonito no Power BI e no Oracle
df_matriz_risco.rename(columns={
    'Cluster': 'ID_PERSONA',
    'Qtd_Incidentes': 'VOLUME_INCIDENTES',
    'NM_GRUPO_DESIGNADO': 'EQUIPE',
    'NM_CATEGORIA': 'CATEGORIA_CRITICA'
}, inplace=True)

df_matriz_risco['NR_DURACAO_HORAS'] = df_matriz_risco['NR_DURACAO_HORAS'].round(0).astype(int)

print("\nTabela de Matriz de Risco criada com sucesso!")
print(df_matriz_risco.head())

In [ ]:
# 1. Definindo as Tipagens Nativas do Oracle
# Tabela 1: Volume Diário (Previsões)
tipagem_volume = {
    'DATA': TIMESTAMP(timezone=False),
    'VOLUME_REAL': NUMBER(precision= 10, scale = 0),
    'PREVISAO_D7': NUMBER(precision= 10, scale = 0),
    'PREVISAO_D1': NUMBER(precision= 10, scale = 0)
}

# Tabela 2: Matriz de Risco (Clusters/Personas)
tipagem_risco = {
    'ID_PERSONA': NUMBER(precision= 10, scale = 0),
    'VOLUME_INCIDENTES': NUMBER(precision= 10, scale = 0),
    'TP_PRIORIDADE':VARCHAR2(50),
    'EQUIPE':VARCHAR2(100),
    'CATEGORIA_CRITICA':VARCHAR2(100),
    'FL_KPI_VIOLADO':VARCHAR2(10),
    'TP_ABERTO_POR':VARCHAR2(50),
    'NR_DURACAO_HORAS': NUMBER(10,0) # Limita a 2 casas decimais no banco
}

# Tabela 3: Feed de Alertas (Guardião do OLA + GenAI)
tipagem_alertas = {
    'CD_INCIDENTE':VARCHAR2(50),
    'DT_ABERTO': TIMESTAMP(timezone=False),
    'TP_PRIORIDADE':VARCHAR2(50),
    'NM_CATEGORIA':VARCHAR2(100),
    'NM_GRUPO_DESIGNADO':VARCHAR2(100),
    'RISCO_SLA_PREVISTO': NUMBER(precision= 10, scale = 0),
    'STATUS_REAL': NUMBER(precision= 10, scale = 0),
    'TEXTO_PRESCRITIVO_LLM':VARCHAR2(1000) # Espaço generoso para a resposta do LLM
}

In [ ]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
CONNECT_STRING = '(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1522)(host=adb.sa-saopaulo-1.oraclecloud.com))(connect_data=(service_name=g253a13c6f2211a_dadosaiopslocaweb_low.adb.oraclecloud.com))(security=(ssl_server_dn_match=yes)))'

def conectar_oracle():
    return oracledb.connect(
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING,
        wallet_location=r"C:/opt/OracleCloud/Wallet_dadosAIOpsLocaweb",
        wallet_password=DB_PASSWORD
    )

In [ ]:
# Criando a conexão
engine = create_engine("oracle+oracledb://", creator = conectar_oracle)
print("Iniciando upload para o Oracle Cloud...")
# Enviando as 3 tabelas para o Oracle Cloud

df_volume.to_sql('TB_PREVISAO_DIARIA', con=engine, if_exists='append', index=False)
print("- TB_PREVISAO_DIARIA inserida com sucesso!")

df_matriz_risco.to_sql('TB_RISCO_EQUIPE', con=engine, if_exists='append', index=False)
print("- TB_RISCO_EQUIPE inserida com sucesso!")

df_alertas_criticos.to_sql('TB_ALERTAS_ITSM', con=engine, if_exists='append', index=False)
print("- TB_ALERTAS_ITSM inserida com sucesso!")

print("Dados enviados para o Oracle Cloud com sucesso!")

In [ ]:
# df_volume.to_csv('TB_PREVISAO_DIARIA.csv', index=False)
# df_matriz_risco.to_csv('TB_RISCO_EQUIPE.csv', index=False)
# df_alertas_criticos.to_csv('TB_ALERTAS_ITSM.csv', index=False)

# print("Arquivos locais gerados com sucesso para o Power BI!")